In [1]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

In [2]:
document = """
Apex Motion's Pune plant manufactures BMS Controller X1. 
BMS Controller X1 requires MCU-900. 
MCU-900 is supplied by Shenzhen Microelectronics. 
Shenzhen Microelectronics depends on Taiwan Silicon Corp for semiconductor wafers. 
Taiwan Silicon Corp operates a facility in Hsinchu. 
The Hsinchu facility was affected by an earthquake. 
Current estimated production delay is 14 days.
"""

In [4]:
document_data = {
    "document":"DOC001",
    "title": "Apex Motion's Pune plant manufactures BMS Controller X1.",
    "source": "Apex Motion's Pune plant",
    "text": document
}

In [5]:
document_data

{'document': 'DOC001',
 'title': "Apex Motion's Pune plant manufactures BMS Controller X1.",
 'source': "Apex Motion's Pune plant",
 'text': "\nApex Motion's Pune plant manufactures BMS Controller X1. \nBMS Controller X1 requires MCU-900. \nMCU-900 is supplied by Shenzhen Microelectronics. \nShenzhen Microelectronics depends on Taiwan Silicon Corp for semiconductor wafers. \nTaiwan Silicon Corp operates a facility in Hsinchu. \nThe Hsinchu facility was affected by an earthquake. \nCurrent estimated production delay is 14 days.\n"}

In [6]:
def run_query(query, parameters=None):
    records, summary, keys = driver.execute_query(query, parameters_=parameters or {})
    return [record.data() for record in records]

In [7]:
query="""create (d:Document {document:$document, title:$title, source:$source, text:$text})"""
run_query(query, parameters=document_data)

[#D390]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.124.169.171', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8015434138027647s (Unable to retrieve routing information)
[#D391]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-27-0122.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))): OSError('No data')
Transaction failed and will be retried in 2.1370470548074647s (Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-27-0122.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))))


[]

In [8]:
chunks = [
    {
        "chunk_id": "C001",
        "text": "Apex Motion's Pune plant manufactures BMS Controller X1."
    },
    {
        "chunk_id": "C002",
        "text": "BMS Controller X1 requires MCU-900."
    },
    {
        "chunk_id": "C003",
        "text": "MCU-900 is supplied by Shenzhen Microelectronics."
    },
    {
        "chunk_id": "C004",
        "text": "Shenzhen Microelectronics depends on Taiwan Silicon Corp for semiconductor wafers."
    },
    {
        "chunk_id": "C005",
        "text": "Taiwan Silicon Corp operates a facility in Hsinchu."
    },
    {
        "chunk_id": "C006",
        "text": "The Hsinchu facility was affected by an earthquake. Current estimated production delay is 14 days."
    }
]

query= """create (c:Chunk {chunk_id:$chunk_id, text:$text})"""
for chunk in chunks:
    run_query(query, parameters=chunk)

In [9]:
query = """MATCH (d:Document {document:$document})
MATCH (c:Chunk {chunk_id:$chunk_id})
MERGE (d)-[:HAS_CHUNK]->(c)
"""
for chunk in chunks:
    run_query(query, parameters={"document": "DOC001", "chunk_id": chunk["chunk_id"]})